# ARC-AGI-2 solver kernel v6s -- theory v33 landing: Sigma-profile solver construction

**What this notebook is.** The NVARC+ v3 triple-pass pooled ensemble solver
(LoRA test-time training + turbo DFS constraint decoding + augmentation
consistency scoring, three independently seeded passes with a 3-store
candidate pool), plus the footprint-matched controller (theory v32 landing,
revision 2) and the P4 cross-group second candidate (theory v32-core section 6,
Campaign B), with run-to-run determinism hardening.

**Theory basis (v33, `ARC-AGI-2-v33-sigma-profile.md`).** Every ARC solver --
including the current top-5 (Tufa Labs, Yi-Chia Chen, rabbithole, nvbanana,
_hans) -- is a point in a five-stage construction profile
(prior / adaptation / generation / aggregation / decision); the top-5 are
extreme faces of the same design space, not different kinds of system. This
notebook implements the neural-adaptation face (LoRA TTT + DFS decoding +
pooled voting + coverage-aware set selection) under a 12 h single-L4 budget,
with the controller acting as an automatic stage-selector.

**v6s changes vs v5:**

1. **Test-set mode.** A new first cell forces the competition-rerun dataset
   (240 hidden test tasks) for API/interactive runs; the submitted
   submission.json always covers the test set.
2. **P4 set selection.** attempt_2 prefers a structurally distinct candidate
   (different output shape or >25% cells different -- a proxy for a different
   rule family, proposition D2) over the marginal runner-up when the
   independent probmul hedge does not fire. attempt_1, the hedge and the
   pass-A top-1 invariant are unchanged. In A/B mode both variants are scored
   on both arms (primary verdict = controller under v3 selection, comparable
   to the v4 A/B; P4 delta = clearly-labelled secondary analysis).
3. Everything else (mu v2 footprint-matched controller: reseed / deepen /
   verify_more / light / standard; determinism hardening; single-build
   discipline; v3 budget partition A 45% / B 78% / C 85-100%) is unchanged
   from v5 and frozen in `ARC-AGI-2-v5-controller-prereg.md`.

**Run profiles** (auto-selected): competition rerun or forced test mode ->
`submission` (12 h, single pre-registered adaptive arm over the pass-C
window); else `ab_validate` (330 min, full A/B on the public evaluation set,
C window split 45/55, submission-aligned fractions).


In [ ]:
# ---------------------------------------------------------------------------
# v6s: dataset mode + theory header (new first code cell).
#
# WHY THIS CELL EXISTS.  The submitted artifact must cover the 240-task hidden
# TEST set (arc-agi_test_challenges.json), but a kernel run started from the
# API / an interactive session is NOT a competition rerun, so every later
# cell's RERUN flag would otherwise select the 120-task public evaluation set
# and produce a submittable-for-nothing submission.json.  Setting the flag
# here -- before any other cell reads it, and inherited by every solver
# subprocess through os.environ -- makes the whole kernel (budget profile,
# task files, checkpoint daemon, controller, selection, final safety net)
# behave exactly as in a competition rerun, on the hidden test set.
#
# Modes (baked at build time by build_kernel_v6s.py):
#   test  : force KAGGLE_IS_COMPETITION_RERUN=1  -> submission profile, 240 tasks
#   eval  : leave the flag unset                 -> ab_validate profile, 120 tasks
#   smoke : leave the flag unset                 -> commit profile, 4 debug tasks
# ---------------------------------------------------------------------------
import os, json

MODE = "test"
if MODE == "test":
    os.environ["KAGGLE_IS_COMPETITION_RERUN"] = "1"

COMP = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2"
TEST_F = f"{COMP}/arc-agi_test_challenges.json"
EVAL_F = f"{COMP}/arc-agi_evaluation_challenges.json"
_rerun = bool(os.getenv("KAGGLE_IS_COMPETITION_RERUN"))
_path = TEST_F if _rerun else EVAL_F
assert os.path.isfile(_path), f"dataset file missing: {_path}"
_tasks = json.load(open(_path))
_n_tasks = len(_tasks)
_n_units = sum(len(t["test"]) for t in _tasks.values())
print(f"[v6s] mode={MODE} rerun={_rerun} dataset={os.path.basename(_path)} "
      f"tasks={_n_tasks} units={_n_units}")
assert _n_tasks == (240 if _rerun else 120), f"unexpected task count {_n_tasks}"


In [ ]:
# ---------------------------------------------------------------------------
# v5: run-to-run reproducibility hardening (new cell, runs before everything).
#
# WHY: the same byte-identical kernel scored 31.94 (10-12) and 27.64 (10-15)
# on the private set -- a 4.3-point rerun noise that dwarfs the controller's
# measured effect (~+2.5 tasks).  All RNG streams are already explicitly
# seeded (lora_seed / train_aug_seed / eval_aug_seed / score_seed_off per
# pass), so the residual noise is (a) CUDA bf16 non-determinism inside the
# Unsloth TTT loop and (b) wall-clock cap jitter.  This cell attacks (a):
#
#   CUBLAS_WORKSPACE_CONFIG  makes cuBLAS use deterministic workspaces
#                            (propagates to every subprocess via os.environ)
#   use_deterministic_algorithms(True, warn_only=True)  is applied inside
#                            each solver worker (see the arc_solver patch in
#                            build_kernel_v5_arc.py) -- warn_only so an op
#                            without a deterministic implementation degrades
#                            to a warning instead of crashing the 12h run.
#
# The env var ARC_DETERMINISTIC=0 disables both (pre-registered contingency:
# if the v5 A/B fixed-arm coverage drops >5% vs the v4 A/B fixed arm,
# deterministic algorithms are costing throughput and get disabled for the
# submission -- prereg v2 section 7).
# ---------------------------------------------------------------------------
import os

os.environ["ARC_DETERMINISTIC"] = os.getenv("ARC_DETERMINISTIC", "1")
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
os.environ.setdefault("PYTHONHASHSEED", "0")

print(f"[determinism] ARC_DETERMINISTIC={os.environ['ARC_DETERMINISTIC']} "
      f"CUBLAS_WORKSPACE_CONFIG={os.environ['CUBLAS_WORKSPACE_CONFIG']} "
      f"PYTHONHASHSEED={os.environ['PYTHONHASHSEED']}")


In [ ]:
# ---------------------------------------------------------------------------
# Global wall-clock budget (v5: three run profiles).
#
#   submission : competition rerun (hidden test set), 12h - 10min, and a single
#                pre-registered arm fills the pass-C window (v3 store).
#   ab_validate: commit-mode full A/B validation on the public evaluation set
#                (all tasks, shared A/B pools, C window split 45/55).
#   commit     : cheap 4-debug-task smoke run; every phase is exercised.
#
# BUDGET RELATIONS (verified 2026-10-17 against the v3 base source, cells 1/10):
#   * The submission/commit partition is byte-identical to v3's (A 45% / B 78% /
#     C_catchup 85% / single-arm C sweep [85%, 100%] = 15% of 12h = 1.8h), so
#     the submitted arm window equals v3's seedC window exactly.
#   * The ab_validate partition (A 28% / B 48% / catchup 52%, fixed C 71min,
#     adaptive C 87min) is a COMPRESSED PROXY: the adaptive arm gets 87min in
#     validation vs 108min in the submission.  Time-hungry interventions
#     (reseed cap 950, deepen cap 1000) are truncated harder in validation,
#     biasing the A/B towards under-estimating the intervention effect
#     (under-powered, never towards false positives).  Declared in prereg v2.
#
# VALIDATE_FULL is a build-time switch baked into the pushed notebook (env
# override kept for local dry-runs only).
# ---------------------------------------------------------------------------
import os, time, json
RERUN = bool(os.getenv("KAGGLE_IS_COMPETITION_RERUN"))
T0 = time.time()
VALIDATE_FULL = os.getenv("ARC_VALIDATE_FULL", "1") == "1"
PROFILE = "submission" if RERUN else ("ab_validate" if VALIDATE_FULL else "commit")
# Pre-registered submission arm (prereg v2 section 6): chosen from the A/B result.
SUBMIT_ARM = os.getenv("ARC_SUBMIT_ARM", "adaptive")

if PROFILE == "submission":
    global_end_time = T0 + 12 * 3600 - 600
elif PROFILE == "ab_validate":
    global_end_time = T0 + float(os.getenv("ARC_VALIDATE_BUDGET_MIN", "330")) * 60
else:
    global_end_time = T0 + 55 * 60

BUDGET = global_end_time - T0
if PROFILE == "ab_validate":
    # submission-aligned fractions: A/B pools share the same phase boundaries
    # as the 12h rerun, both C arms get a real window (45/55 of the C window)
    PHASE = dict(A=T0 + 0.28 * BUDGET, B=T0 + 0.48 * BUDGET,
                 C_catchup=T0 + 0.52 * BUDGET, C=global_end_time)
    T_FIXED = T0 + 0.52 * BUDGET + 0.45 * 0.48 * BUDGET
else:
    PHASE = dict(A=T0 + 0.45 * BUDGET, B=T0 + 0.78 * BUDGET,
                 C_catchup=T0 + 0.85 * BUDGET, C=global_end_time)
    T_FIXED = T0 + 0.85 * BUDGET + 0.45 * 0.15 * BUDGET

if PROFILE == "ab_validate":
    os.environ["ARC_ALL_KEYS"] = "1"
    os.environ["ARC_CKPT_C_STORE"] = "/kaggle/inference_outputs_c_adaptive"
os.makedirs("/kaggle/working/logs", exist_ok=True)

def phase_left(name):
    return (PHASE[name] - time.time()) / 60.0

print(f"profile={PROFILE} rerun={RERUN} budget={(global_end_time-T0)/3600:.2f}h "
      f"submit_arm={SUBMIT_ARM} validate_full={VALIDATE_FULL}")
print("phase deadlines (minutes from now): "
      + ", ".join(f"{k}={phase_left(k):.1f}" for k in PHASE)
      + ("" if PROFILE == "submission" else f", T_fixed={(T_FIXED-time.time())/60:.1f}"))
if PROFILE == "ab_validate":
    print(f"adaptive-arm C window share: {100*(1-(T_FIXED-T0)/BUDGET):.1f}% of budget "
          f"(compressed proxy: the submitted single arm gets 108 min of the 12 h run)")


In [ ]:
!pip uninstall -y tensorflow
import torch, sys, os
print(sys.version.split()[0], "torch", torch.__version__, "gpus", torch.cuda.device_count(),
      [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
import glob
print("models:", glob.glob("/kaggle/input/models/*/*"), glob.glob("/kaggle/input/qwen*"))
print("utility scripts:", [p for p in sys.path if "/kaggle/usr/lib" in p][:5])
os.makedirs("/kaggle/working/logs", exist_ok=True)


In [ ]:
%%writefile arc_loader.py
import json
import numpy as np
from transformers import AutoTokenizer


def convert_grid_to_string(grid) -> str:
    text = ""
    for row in grid:
        for cell in row:
            text += str(int(cell))
        text += "\n"
    return text.strip()

def is_valid_solution(guess):
    return isinstance(guess, np.ndarray) and guess.ndim == 2 and all(0 < x <= 30 for x in guess.shape)

def shuffled(data_list):
    return np.random.permutation(data_list).tolist()

def permute_mod(a, descriptor, invert=False):
    permutation = [int(i) for i in descriptor if str(i).isdigit()]
    assert sorted(permutation)==list(range(10))
    a = np.asarray(a)
    if a.ndim==3:
        if not invert: permutation = np.argsort(permutation)
        a = a[..., permutation]
    else:
        assert a.ndim==2
        if invert: permutation = np.argsort(permutation)
        a = np.asarray(permutation)[a]
    return a

def permute_rnd_all_(query):
    permutation = np.random.permutation(10).tolist()
    return 'permute' + ''.join(map(str, permutation))


class QwenFormatter:

    def __init__(self, tokenizer: AutoTokenizer):
        self.tokenizer = tokenizer

    def fmt_query(self, query) -> str:
        grid_input = convert_grid_to_string(query[0]["input"])
        return "<|im_start|>user\n" + grid_input + "<|im_end|><|im_start|>assistant\n"

    def fmt_reply(self, reply) -> str:
        return convert_grid_to_string(reply[0]) + "<|im_end|>"

    def fmt_train(self, train, last_is_challenge=False) -> str:
        if last_is_challenge:
            test = train[-1]
            train = train[:-1]
        else:
            test = None
        text = ""
        for x in train:
            grid_input = convert_grid_to_string(x["input"])
            grid_output = convert_grid_to_string(x["output"])
            text += f"<|im_start|>user\n{grid_input}<|im_end|><|im_start|>assistant\n{grid_output}<|im_end|>"
        if test is not None:
            text += self.fmt_query([test]) + self.fmt_reply([test["output"]])
        return text

    def max_new_tokens(self):
        max_sized_reply = np.zeros([30, 30], dtype=int)
        tokens = self.tokenizer.encode(self.fmt_reply([max_sized_reply]))
        return len(tokens) + 1

    def convert_tokens_to_array(self, tokens, limit_rows=30):
        if len(tokens) < 2:
            return None
        text = self.tokenizer.decode(tokens[:-1])
        try:
            lines = text.strip().split("\n")
            by_rows = [row for row in [[int(x) for x in line if x.isdigit()] for line in lines] if len(row)]
            if len(by_rows) > limit_rows:
                by_rows = by_rows[:limit_rows]
            array = np.array(by_rows, dtype=int)
            if is_valid_solution(array):
                return array
        except:
            pass
        return None


class ArcDataset:

    @staticmethod
    def forward_mod(a, key, use_perm=True):
        if a is None: return a
        for op in key.split('.')[1:]:
            if   op=='rot90':              a = np.rot90(a)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=False) if use_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    @staticmethod
    def invert_mod(a, key, inv_perm=True):
        if a is None: return a
        for op in key.split('.')[1:][::-1]:
            if   op=='rot90':              a = np.rot90(a, k=3)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=True) if inv_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    def __init__(self, queries, replies={}, keys=None, is_orig=False):
        if keys is not None: keys = [k for k in keys if k is not None]
        self.queries = queries if keys is None else {k: queries[k] for k in keys}
        self.replies = replies if keys is None else {k: replies[k] for k in keys if k in replies}
        self.is_orig = is_orig
        self.keys = sorted(queries.keys()) if keys is None else keys
        self.transposed_dataset = None

    def change_keys(self, keys, keep_flags=False):
        flags = dict(is_orig=self.is_orig) if keep_flags else {}
        return self.__class__(queries=self.queries, replies=self.replies, keys=keys, **flags)

    @classmethod
    def from_file(cls, queries_file, keys=None):
        with open(queries_file) as f:
            queries = f.read()
        return cls(
            queries=json.loads(queries),
            is_orig=True,
            keys=keys,
        )

    def load_replies(self, replies_file):
        print(f"*** Load solutions from '{replies_file}'...")
        with open(replies_file) as f: replies = f.read()
        replies_parsed = json.loads(replies)
        self.replies = {k: replies_parsed[k] for k in self.keys}
        return self

    def split_multi_replies(self):
        key_indices = [(k, i) for k in self.keys for i in range(len(self.queries[k]['test']))]
        return self.__class__(
            keys=[f'{k}_{i}' for k, i in key_indices],
            queries={f'{k}_{i}': {'train': self.queries[k]['train'], 'test': [self.queries[k]['test'][i]]} for k, i in key_indices},
            replies={f'{k}_{i}': [self.replies[k][i]] for k, i in key_indices if k in self.replies},
        )

    def shuffled(self):
        return self.__class__(queries=self.queries, replies=self.replies, keys=shuffled(self.keys))

    def append(*datasets):
        return datasets[0].__class__(
            queries={k: v for d in datasets for k, v in d.queries.items()},
            replies={k: v for d in datasets for k, v in d.replies.items()},
            keys   =[k    for d in datasets for k    in d.keys           ],
        )

    def mod_single(self, mod_func, descriptor, i, keep_key, inputs_only):
        queries = {}
        replies = {}
        keys    = []
        for k0 in self.keys:
            desc = (('copy{i}' if mod_func is np.copy else mod_func.__name__) if descriptor is None else descriptor if isinstance(descriptor, str) else descriptor(self.queries[k0])).format(i=i)
            func = lambda a, d: np.asarray(mod_func(a) if descriptor is None else mod_func(a, d)).tolist()
            k1 = k0 if keep_key else f"{k0}.{'I' if inputs_only else ''}{desc}"
            keys.append(k1)
            queries[k1] = {m: [{t: (func(a, desc) if t=='input' or not inputs_only else a) for t, a in x.items()} for x in e] for m, e in self.queries[k0].items()}
            if k0 in self.replies:
                replies[k1] = [func(a, desc) for a in self.replies[k0]]
        ret = self.__class__(queries=queries, replies=replies, keys=keys)
        return ret

    def mod(self, mod_func, descriptor=None, n=1, stack=None, keep=False, keep_key=False, shuffle=False, join=True, inputs_only=False):
        assert not (keep and keep_key)
        cur = self
        ret = [cur.shuffled() if shuffle else cur] if keep else []
        if stack is None: stack = mod_func.__name__.startswith('rot')
        for i in range(n):
            cur = (cur if stack else self).mod_single(mod_func, descriptor, i=i, keep_key=keep_key, inputs_only=inputs_only)
            ret.append(cur.shuffled() if shuffle else cur)
        return self.__class__.append(*ret) if join else ret

    def get(self, key, formatter: QwenFormatter):
        train = formatter.fmt_train(self.queries[key]['train'])
        query = formatter.fmt_query(self.queries[key]['test'])
        reply = formatter.fmt_reply(self.replies[key]) if key in self.replies else ''
        text = train+query+reply if reply else formatter.fmt_train(self.queries[key]['train'], last_is_challenge=True)
        return dict(key=key, train=train, query=query, reply=reply, input=train+query, text=text)

    def as_list(self, formatter: QwenFormatter):
        return [self.get(key, formatter) for key in self.keys]

    def get_length(self, key, formatter: QwenFormatter, name, max_of_transposed=False):
        if formatter is None:
            if   name=='input': return sum(np.prod(np.shape(v)) for v3 in self.queries[key].values() for v2 in v3 for v in v2.values())
            elif name=='reply': return sum(np.prod(np.shape(v)) for v in self.replies[key])
            else: assert False
        else:
            datasets = [self]
            if max_of_transposed:
                if self.transposed_dataset is None: self.transposed_dataset = self.mod(np.transpose, keep=False, keep_key=True)
                datasets.append(self.transposed_dataset)
            return max(len(formatter.tokenizer.encode(ds.get(key, formatter=formatter)[name])) for ds in datasets)

    def cut_to_len(self, formatter, name, max_len, from_end=False):
        temp_ds = self.change_keys(self.keys)
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            reply = temp_ds.replies.get(key)
            while max_len<temp_ds.get_length(key, formatter=formatter, name=name):
                query = temp_ds.queries[key]
                if not key.split('.')[-1].startswith('ex'):
                    key = f"{key}.ex{''.join(map(str, range(len(query['train']))))}"
                key_split = key.split('.')
                assert key_split[-1].startswith('ex')
                key = '.'.join(key_split[:-1] + [f'ex{key_split[-1][2:-1] if from_end else key_split[-1][3:]}'])
                temp_ds.queries[key] = {k: ((v[:-1] if from_end else v[1:]) if k=='train' else v) for k, v in query.items()}
                if reply is not None:
                    temp_ds.replies[key] = reply
            new_keys.append(key)
            new_queries[key] = temp_ds.queries[key]
            if reply is not None: new_replies[key] = reply
        return self.__class__(keys=new_keys, queries=new_queries, replies=new_replies)
    
    def shuffle_ex(self, perm=None, keep_max=None):
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            n = len(self.queries[key]['train'])
            p = np.random.permutation(n) if perm is None else perm
            if keep_max is not None: p = p[:keep_max]
            new_key = f'{key}.ex' + ('-' if (p.max()>9) else '').join(map(str, p.tolist()))
            new_keys.append(new_key)
            new_queries[new_key] = {k: (np.array(v, dtype=object)[p].tolist() if k=='train' else v) for k, v in self.queries[key].items()}
            if key in self.replies: new_replies[new_key] = self.replies[key]
        return self.__class__(queries=new_queries, replies=new_replies, keys=new_keys)

    def augment(self, n=1, shfl_keys=False, seed=42):
        np.random.seed(seed)
        d = self
        d = d.mod(np.transpose, keep=True)
        d = d.mod(np.rot90, n=3, keep=True)
        d = d.mod(permute_mod, permute_rnd_all_, n=n, shuffle=shfl_keys, keep=False)
        d = d.shuffle_ex()
        return d

    def get_submission(self, results=None):
        assert self.is_orig==True, 'Must be run on original dataset.'
        submission = {k: [{f'attempt_{i+1}': [[0]] for i in range(2)} for _ in range(len(self.queries[k]['test']))] for k in self.keys}
        if results is not None: self.fill_submission(results, submission)
        return submission

    @staticmethod
    def fill_submission(results, submission):
        print(f'*** Generating submission for {len(results)} outputs...')
        for k, v in results.items():
            base_id, base_nr = k.split('_')
            target_dict = submission[base_id][int(base_nr)]
            for i, g in enumerate(v[:len(target_dict)]):
                target_dict[f'attempt_{i+1}'] = g.tolist()

    def validate_submission(self, submission):
        assert self.is_orig==True, 'Must be run on original dataset.'
        score = 0
        for k, v in self.replies.items():
            for i, r in enumerate(v):
                for attempt in ['attempt_1', 'attempt_2']:
                    if np.array_equal(r, submission[k][i][attempt]):
                        score += 1 / len(v)
                        break
        return score

In [ ]:
%%writefile arc_decoder.py
import os
import bz2
import pickle
import numpy as np

def hashable(guess):
    return tuple(map(tuple, guess))

def score_sum(guesses, getter):
    guess_list = list(guesses.values())
    scores = {}
    for g in guess_list:
        h = hashable(g["solution"])
        x = scores[h] = scores.get(h, [[], g["solution"]])
        x[0].append(g)
    scores = [(getter(sc), o) for sc, o in scores.values()]
    scores = sorted(scores, key=(lambda x: x[0]), reverse=True)
    ordered_outputs = [x[-1] for x in scores]
    return ordered_outputs

def getter_full_probmul_3(guesses, baseline=3):
    inf_score = np.sum([baseline-g["beam_score"] for g in guesses])
    aug_score = np.mean([np.sum([baseline-s for s in g["score_aug"]]) for g in guesses])
    return inf_score + aug_score

def score_full_probmul_3(guesses):
    return score_sum(guesses, getter_full_probmul_3)

def getter_kgmon(guesses):
    inf_score = len(guesses)
    aug_score = np.mean([np.mean(g["score_aug"]) for g in guesses])
    return inf_score - aug_score

def score_kgmon(guesses):
    return score_sum(guesses, getter_kgmon)


selection_algorithms = [
    score_full_probmul_3,
    score_kgmon,
]


def _valid_sample(sample):
    try:
        sol = np.asarray(sample["solution"])
        if sol.ndim != 2 or sol.size == 0 or sol.shape[0] > 30 or sol.shape[1] > 30:
            return False
        if not np.issubdtype(sol.dtype, np.integer):
            return False
        if sol.min() < 0 or sol.max() > 9:
            return False
        if not np.isfinite(sample["beam_score"]):
            return False
        if not len(sample["score_aug"]) or not np.all(np.isfinite(sample["score_aug"])):
            return False
        return True
    except Exception:
        return False


class ArcDecoder:

    def __init__(self, dataset, n_guesses):
        self.dataset = dataset
        self.n_guesses = n_guesses
        self.decoded_results = {}
        try:
            self.valid_keys = set(dataset.keys)
        except Exception:
            self.valid_keys = None

    def load_decoded_results(self, store, run_name=""):
        if not os.path.isdir(store):
            print(f"*** No decoded results at {store}")
            return 0
        n_files = n_samples = n_bad = 0
        for key in os.listdir(store):
            try:
                with bz2.BZ2File(os.path.join(store, key)) as f:
                    outputs = pickle.load(f)
            except Exception as e:
                print(f"*** Skipping corrupt shard {key}: {e}")
                n_bad += 1
                continue
            n_files += 1
            base_key = key.split(".")[0]
            if self.valid_keys is not None and base_key not in self.valid_keys:
                n_bad += 1
                continue
            for i, sample in enumerate(outputs):
                if not _valid_sample(sample):
                    n_bad += 1
                    continue
                self.decoded_results.setdefault(base_key, {})[f"{key}{run_name}.out{i}"] = sample
                n_samples += 1
        print(f"*** Loaded {n_files} shards / {n_samples} samples from {store} (skipped {n_bad})")
        return n_samples

    def candidate_stats(self):
        """Per output key: number of unique candidate grids, number of samples, and the
        vote count / kgmon margin of the leading candidate (how sure the pool is)."""
        stats = {}
        for bk, v in self.decoded_results.items():
            groups = {}
            for g in v.values():
                groups.setdefault(hashable(g["solution"]), []).append(g)
            scored = sorted((getter_kgmon(gs) for gs in groups.values()), reverse=True)
            top_votes = max(len(gs) for gs in groups.values())
            margin = (scored[0] - scored[1]) if len(scored) > 1 else float("inf")
            stats[bk] = dict(unique=len(groups), samples=len(v), top_votes=top_votes, margin=margin)
        return stats

    def run_selection_algo(self, selection_algorithm=score_kgmon):
        return {bk: selection_algorithm({k: g for k, g in v.items()}) for bk, v in self.decoded_results.items()}

    def benchmark_selection_algos(self):
        print("*** Benchmark selection algorithms...")

        labels = {}
        num_tasks_per_puzzle = {}
        num_solved_keys = 0
        num_total_keys = 0

        correct_beam_scores = []

        for basekey, basevalues in self.decoded_results.items():

            mult_key, mult_sub = basekey.split("_")
            num_tasks_per_puzzle[mult_key] = max(num_tasks_per_puzzle.get(mult_key, 0), int(mult_sub) + 1)

            labels[basekey] = correct_solution = self.dataset.replies[basekey][0]

            for subkey, sample in basevalues.items():

                solution = sample["solution"]
                beam_score = sample["beam_score"]
                aug_mean = np.mean(sample["score_aug"])

                if np.shape(correct_solution) != np.shape(solution):
                    corr_str = "bad_xy_size"
                elif np.array_equal(correct_solution, solution):
                    corr_str = "ALL_CORRECT"
                    num_solved_keys += 1
                    correct_beam_scores.append(beam_score)
                else:
                    corr_str = "bad_content"

                output_len = f"{solution.shape[0]}x{solution.shape[1]}"

                if corr_str == "ALL_CORRECT":
                    print(f"{corr_str}:{beam_score:8.5f} - {aug_mean:8.5f} {output_len:5s} [{subkey}]")
                num_total_keys += 1

        print(f" subkeys: {num_solved_keys}/{num_total_keys}")
        if correct_beam_scores:
            print(f" avg correct beam score: {np.mean(correct_beam_scores):8.5f}")
            print(f" max correct beam score: {np.max(correct_beam_scores):8.5f}")

        num_puzzles = len(num_tasks_per_puzzle)

        for selection_algorithm in selection_algorithms:
            name = selection_algorithm.__name__
            selected = self.run_selection_algo(selection_algorithm)
            correct_puzzles = {k for k, v in selected.items() if any(np.array_equal(guess, labels[k]) for guess in v[:self.n_guesses])}
            print(correct_puzzles)
            score = sum(1/num_tasks_per_puzzle[k.split("_")[0]] for k in correct_puzzles)
            print(f" acc: {score:5.1f}/{num_puzzles:3} ('{name}')")


In [ ]:
%%writefile arc_solver.py
from unsloth import FastLanguageModel, UnslothTrainingArguments, UnslothTrainer
from arc_loader import ArcDataset, QwenFormatter

import gc
import os
import io
import time
import zlib
import json
import torch
import numpy as np
from tqdm import tqdm
from datasets import Dataset
from collections import defaultdict

from typing import Any, Union
from transformers import DataCollatorForLanguageModeling

import logging
from contextlib import redirect_stdout, redirect_stderr

from peft import get_peft_model_state_dict, set_peft_model_state_dict

import bz2
import pickle
import traceback

logging.disable(logging.WARNING)

# ---------------------------------------------------------------------------
# Run configuration (all overridable through environment variables so the
# same solver can run the LB-proven primary pass and a wider secondary pass).
# Defaults == the public LB 33.89 NVARC configuration.
# ---------------------------------------------------------------------------
def _env(name, default, cast):
    v = os.getenv(name)
    return cast(v) if v not in (None, "") else default

CFG = dict(
    model_path      = _env("ARC_MODEL_PATH", "", str),
    out_dir         = _env("ARC_OUT_DIR", "/kaggle/inference_outputs", str),
    lora_seed       = _env("ARC_LORA_SEED", 42, int),
    train_aug_seed  = _env("ARC_TRAIN_AUG_SEED", 1, int),
    n_train_aug     = _env("ARC_N_TRAIN_AUG", 16, int),     # x8 geometries = 128 sequences
    num_epochs      = _env("ARC_EPOCHS", 1, int),
    learning_rate   = _env("ARC_LR", 5e-5, float),
    eval_aug_seed   = _env("ARC_EVAL_AUG_SEED", 2, int),
    n_eval_aug      = _env("ARC_N_EVAL_AUG", 2, int),       # x8 geometries = 16 decoded views
    min_prob        = _env("ARC_MIN_PROB", 0.2, float),     # DFS cumulative prob threshold
    dfs_window      = _env("ARC_DFS_WINDOW", 540.0, float), # seconds per DFS call
    task_cap        = _env("ARC_TASK_CAP", 1200.0, float),  # seconds per task (decode phase)
    score_seed_off  = _env("ARC_SCORE_SEED_OFFSET", 0, int),
    decode_batch    = _env("ARC_DECODE_BATCH", 4, int),
)
# ---------------------------------------------------------------------------
# v5 (theory v32 landing, revision 2): per-task configuration overrides +
# run telemetry + model-side intervention keys.
# The controller cell writes {task_key: {cfg_key: value}} to ARC_TASK_CONFIG_FILE;
# each worker applies the overrides for the task it is about to run and restores
# the process defaults afterwards.  Telemetry (one JSON line per task) records
# elapsed time, whether the task hit its cap (the search-pressure signal), the
# shard count and the applied overrides -- the real-ARC diagnostic source.
#
# v5: train_aug_seed / n_train_aug join the overridable keys.  These are read
# inside the task loop (puzzle_ds.augment(..., seed=CFG["train_aug_seed"])),
# so per-task overrides take effect: an empty candidate pool (express == 0)
# can be given a DIFFERENT trained model, not just more decoding effort --
# the footprint-matched intervention for a model-side bottleneck.
# ---------------------------------------------------------------------------
_TASK_CFG_KEYS = ("task_cap", "n_eval_aug", "min_prob", "dfs_window",
                  "score_seed_off", "eval_aug_seed", "train_aug_seed",
                  "n_train_aug")

def _load_task_configs():
    path = os.getenv("ARC_TASK_CONFIG_FILE", "")
    if not path or not os.path.isfile(path):
        return {}
    try:
        with open(path) as f:
            raw = json.load(f)
    except Exception:
        return {}
    out = {}
    for k, v in raw.items():
        if isinstance(v, dict):
            out[k] = {kk: vv for kk, vv in v.items() if kk in _TASK_CFG_KEYS}
    return out

_TASK_CFG = _load_task_configs()

def _apply_task_overrides(key):
    ov = _TASK_CFG.get(key)
    if not ov:
        return {}
    applied = {}
    for k in _TASK_CFG_KEYS:
        if k in ov:
            applied[k] = CFG[k]
            CFG[k] = ov[k]
    return applied

def _restore_task_overrides(applied):
    for k, v in applied.items():
        CFG[k] = v

_TEL_PATH = os.getenv("ARC_TELEMETRY_FILE", "")

def _tel(rec):
    if not _TEL_PATH:
        return
    try:
        with open(_TEL_PATH, "a") as f:
            f.write(json.dumps(rec) + "\n")
    except Exception:
        pass

ARC_VOCAB = {
    "0": 0,
    "1": 1,
    "2": 2,
    "3": 3,
    "4": 4,
    "5": 5,
    "6": 6,
    "7": 7,
    "8": 8,
    "9": 9,
    "Ċ": 10,
    "<|im_end|>": 15,
}

ARC_TOKENS = list(ARC_VOCAB.values())
USER_TOKEN_ID = 11
ASSISTANT_TOKEN_ID = 12
PAD_ID = 13
EOS_ID = 15


def resolve_model_dir():
    if CFG["model_path"] and os.path.isdir(CFG["model_path"]):
        return CFG["model_path"]
    candidates = [
        "/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1",
        "/kaggle/input/qwen3_4b_grids15_sft139/transformers/bfloat16/1",
        "/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/Transformers/bfloat16/1",
    ]
    for c in candidates:
        if os.path.isfile(os.path.join(c, "config.json")):
            return c
    import glob
    for c in glob.glob("/kaggle/input/**/config.json", recursive=True):
        d = os.path.dirname(c)
        if "grids15" in d and os.path.isfile(os.path.join(d, "tokenizer.json")):
            return d
    raise FileNotFoundError("qwen3_4b_grids15_sft139 model directory not found under /kaggle/input")


def stable_seed(key, offset=0):
    return (zlib.crc32(key.encode("utf-8")) + offset) % (1024 ** 2)


class UnslothFixedTrainer(UnslothTrainer):

    # Issue https://github.com/unslothai/unsloth/issues/2435

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        """Fixed compute_loss that handles Unsloth's view tensor issue"""
        if self.label_smoother is not None and "labels" in inputs:
            labels = inputs.pop("labels")
        else:
            labels = None
        outputs = model(**inputs)
        if labels is not None:
            unwrapped_model = self.accelerator.unwrap_model(model)
            if hasattr(unwrapped_model, "_get_name") and "unsloth" in unwrapped_model._get_name().lower():
                loss = self.label_smoother(outputs, labels, shift_labels=True)
            else:
                loss = self.label_smoother(outputs, labels)
        else:
            loss = outputs["loss"] if isinstance(outputs, dict) else outputs[0]
        if hasattr(loss, "clone"):
            loss = loss.clone()
        if self.accelerator.num_processes > 1:
            loss = loss * self.accelerator.num_processes
        return (loss, outputs) if return_outputs else loss


class QwenDataCollatorForCompletionOnlyLM(DataCollatorForLanguageModeling):

    def torch_call(self, examples: list[Union[list[int], Any, dict[str, Any]]]) -> dict[str, Any]:
        batch = super().torch_call(examples)
        for i in range(len(examples)):
            labels = batch["input_ids"][i].clone()
            user_start_idx = np.where(labels == USER_TOKEN_ID)[0].tolist()
            assistant_start_idx = np.where(labels == ASSISTANT_TOKEN_ID)[0].tolist()
            start_idx = sorted(user_start_idx + assistant_start_idx)
            end_idx = np.where(labels == EOS_ID)[0]
            batch["labels"][i, :] = -100
            for j, (start, end) in enumerate(zip(start_idx, end_idx)):
                assert start < end
                if j % 2 == 1:
                    start += 2
                    end += 1
                    batch["labels"][i, start:end] = labels[start:end]
        return batch


# Throughput patch (algebraically identical to the baseline): only the 12
# ARC-token NLL values are moved to the CPU instead of the whole vocabulary.
_ARC_TOKEN_ID_CACHE = {}


def _arc_token_ids(device):
    key = str(device)
    token_ids = _ARC_TOKEN_ID_CACHE.get(key)
    if token_ids is None:
        token_ids = torch.tensor(ARC_TOKENS, dtype=torch.long, device=device)
        _ARC_TOKEN_ID_CACHE[key] = token_ids
    return token_ids


def turbo_dfs(model, logits, max_new_tokens, max_score, scores, pos, cache, start_time, end_time, dfs_window) -> dict:

    n = logits.size(0)

    logits_f = logits.float()
    token_ids = _arc_token_ids(logits.device)
    arc_logits = logits_f.index_select(-1, token_ids)
    nll = (
        torch.as_tensor(scores, dtype=torch.float32, device=logits.device).view(n, 1)
        + torch.logsumexp(logits_f, dim=-1, keepdim=True)
        - arc_logits
    ).cpu()

    suffixes = defaultdict(list)

    candidates = dict()

    for i in range(n):
        candidates[i] = []
        for token_idx, t in enumerate(ARC_TOKENS):
            score = nll[i, token_idx].item()
            if score < max_score:
                if t == EOS_ID:
                    suffixes[i].append((score, [t]))
                elif max_new_tokens > 1:
                    candidates[i].append((score, t))

    for i in range(n):
        candidates[i] = sorted(candidates[i], key=lambda x:x[0])

    while time.time() - start_time < dfs_window and time.time() < end_time:

        batch_tokens = []
        batch_scores = []
        num_alive_beams = 0

        for i in range(n):
            if len(candidates[i]) == 0:
                batch_tokens.append(PAD_ID)
                batch_scores.append(1000)
            else:
                score, t = candidates[i].pop(0)
                batch_tokens.append(t)
                batch_scores.append(score)
                num_alive_beams += 1

        if num_alive_beams == 0:
            break

        outputs = model(
            input_ids=torch.tensor(batch_tokens, device=model.device, dtype=torch.long).view(-1, 1),
            position_ids=torch.full((n, 1), pos, device=model.device),
            past_key_values=cache,
            return_dict=True,
            use_cache=True,
        )

        next_suffixes = turbo_dfs(
            model,
            logits=outputs.logits[:, -1],
            max_new_tokens=max_new_tokens-1,
            max_score=max_score,
            scores=batch_scores,
            pos=pos+1,
            cache=outputs.past_key_values,
            start_time=start_time,
            end_time=end_time,
            dfs_window=dfs_window,
        )

        for batch_id, beams in next_suffixes.items():
            for score, suffix_tokens in beams:
                suffix_tokens.insert(0, batch_tokens[batch_id])
                suffixes[batch_id].append((score, suffix_tokens))

    return suffixes


@torch.no_grad()
def inference_turbo_dfs(model, prefix_tokens, max_new_tokens, max_score, end_time, dfs_window):
    input_ids = torch.tensor(prefix_tokens, device=model.device, dtype=torch.long)
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=True)
    suffixes = turbo_dfs(
        model,
        logits=outputs.logits[:, -1],
        max_new_tokens=max_new_tokens,
        max_score=max_score,
        scores=[0.0] * input_ids.size(0),
        pos=input_ids.size(1),
        cache=outputs.past_key_values,
        start_time=time.time(),
        end_time=end_time,
        dfs_window=dfs_window,
    )
    result = []
    for batch_id, beams in suffixes.items():
        sorted_beams = sorted(beams, key=lambda x:x[0])
        result.append((batch_id, sorted_beams))
    return result


@torch.no_grad()
def calc_scores(queries, answers, tokenizer, model):
    batch_query_tokens = []
    batch_answer_tokens = []
    batch_tokens = []
    batch_lengths = []
    for query, answer in zip(queries, answers):
        query_tokens = tokenizer.encode(query)
        answer_tokens = tokenizer.encode(answer)
        tokens = query_tokens + answer_tokens
        batch_query_tokens.append(query_tokens)
        batch_answer_tokens.append(answer_tokens)
        batch_tokens.append(tokens)
        batch_lengths.append(len(tokens))
    max_len = max(batch_lengths)
    padded_tokens = []
    for tokens in batch_tokens:
        padded = tokens + [PAD_ID] * (max_len - len(tokens))
        padded_tokens.append(padded)
    input_ids = torch.tensor(padded_tokens, device=model.device, dtype=torch.long)

    outputs = model(input_ids=input_ids, return_dict=True, use_cache=False)
    batch_logits = outputs.logits.float()
    batch_log_norm = torch.logsumexp(batch_logits, dim=-1)
    result = []
    for row_id, (query_tokens, answer_tokens) in enumerate(zip(batch_query_tokens, batch_answer_tokens)):
        query_length = len(query_tokens)
        answer_length = len(answer_tokens)
        positions = torch.arange(
            query_length - 1,
            query_length - 1 + answer_length,
            device=model.device,
        )
        target_tokens = torch.tensor(answer_tokens, device=model.device, dtype=torch.long)
        answer_log_probs = (
            batch_logits[row_id, positions, target_tokens]
            - batch_log_norm[row_id, positions]
        )
        result.append(-answer_log_probs.sum().item())
    return result


def make_view_batches(eval_ds, n_perm, batch_size):
    """Group decoded views into batches. With n_perm=2 / batch_size=4 this is
    byte-for-byte the baseline batching ([id,rot180],[rot90,rot270],[T,T.rot180],[T.rot90,T.rot270])."""
    test_id_to_subkeys = defaultdict(list)
    for subkey in sorted(eval_ds.keys):
        test_id = subkey.split(".")[0].split("_")[1]
        test_id_to_subkeys[test_id].append(subkey)
    # sorted geometry order: 0 id, 1 rot90, 2 rot180, 3 rot270, 4 T, 5 T.rot90, 6 T.rot180, 7 T.rot270
    groups_a = [0, 2, 1, 3]          # untransposed geometries
    groups_b = [4, 6, 5, 7]          # transposed geometries
    batches = []
    for geos in (groups_a, groups_b):
        for test_id, subkeys in test_id_to_subkeys.items():
            if n_perm == 2 and batch_size == 4:
                # exact baseline batches: [id,rot180], [rot90,rot270], ...
                for a, b in ((geos[0], geos[1]), (geos[2], geos[3])):
                    batches.append(subkeys[a*n_perm:(a+1)*n_perm] + subkeys[b*n_perm:(b+1)*n_perm])
            else:
                # Batched DFS needs equal token lengths in a batch. Within a geometry group the
                # shape classes are: rot0/rot180 = H x W, rot90/rot270 = W x H, so pair them.
                for a, b in ((geos[0], geos[1]), (geos[2], geos[3])):
                    views = subkeys[a*n_perm:(a+1)*n_perm] + subkeys[b*n_perm:(b+1)*n_perm]
                    for i in range(0, len(views), batch_size):
                        batches.append(views[i:i+batch_size])
    return batches


def worker(rank, queue, end_time, test_path=None):

    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

    if os.getenv("ARC_DETERMINISTIC", "1") == "1":
        try:
            torch.use_deterministic_algorithms(True, warn_only=True)
        except Exception:
            pass

    peft_params = dict(
        r=256,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj", "embed_tokens", "lm_head"],
        lora_alpha=32,
        lora_dropout=0.0,
        bias="none",
        use_gradient_checkpointing=False,
        random_state=CFG["lora_seed"],
        use_rslora=True,
        loftq_config=None,
    )

    train_args = dict(
        per_device_eval_batch_size=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        num_train_epochs=CFG["num_epochs"],
        warmup_steps=0,
        warmup_ratio=0.1,
        max_grad_norm=1.0,
        learning_rate=CFG["learning_rate"],
        optim="adamw_torch",
        weight_decay=0.0,
        lr_scheduler_type="cosine",
        seed=CFG["lora_seed"],
        report_to="none",
        save_strategy="no",
        eval_strategy="no",
        logging_strategy="no",
        fp16=False,
        bf16=True,
        fsdp="",
        ddp_find_unused_parameters=False,
        dataloader_num_workers=0,
        gradient_checkpointing=False,
    )

    max_seq_length = 8192

    model_dir = resolve_model_dir()
    print(f"[Rank {rank}] model dir: {model_dir}")
    print(f"[Rank {rank}] config: {CFG}")

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_dir,
        full_finetuning=False,
        load_in_4bit=False,
        local_files_only=True,
        use_gradient_checkpointing=False,
        max_seq_length=max_seq_length,
    )

    model = FastLanguageModel.get_peft_model(model, **peft_params)

    for name, param in model.named_parameters():
        if param.dtype == torch.float32:
            param.data = param.data.to(torch.bfloat16)

    default_weights = get_peft_model_state_dict(model, adapter_name="default")
    default_weights = {k: v.clone().detach() for k, v in default_weights.items()}

    collator = QwenDataCollatorForCompletionOnlyLM(
        tokenizer=tokenizer,
        mlm=False,
    )

    formatter = QwenFormatter(tokenizer=tokenizer)

    max_new_tokens = formatter.max_new_tokens()

    if test_path is None:
        if rerun_mode:
            test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
        else:
            test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"

    arc_test_set = ArcDataset.from_file(test_path)

    dir_outputs = CFG["out_dir"]
    os.makedirs(dir_outputs, exist_ok=True)

    while True:

        if time.time() > end_time:
            print(f"[Rank {rank}] stop!")
            break

        key = queue.get()
        if key is None:
            break

        start_time = time.time()

        applied = _apply_task_overrides(key)
        max_score = -np.log(CFG["min_prob"])
        hit_cap = False

        try:
            torch.cuda.reset_peak_memory_stats()

            set_peft_model_state_dict(
                model,
                default_weights.copy(),
                adapter_name="default",
            )

            model = FastLanguageModel.for_training(model)

            puzzle_ds = arc_test_set.change_keys([key])

            train_ds = puzzle_ds.augment(n=CFG["n_train_aug"], shfl_keys=True, seed=CFG["train_aug_seed"])
            train_ds = train_ds.cut_to_len(formatter=formatter, name="text", max_len=max_seq_length)

            with io.StringIO() as buf, redirect_stdout(buf), redirect_stderr(buf):

                trainer = UnslothFixedTrainer(
                    model=model,
                    tokenizer=tokenizer,
                    data_collator=collator,
                    train_dataset=Dataset.from_list(train_ds.as_list(formatter)),
                    dataset_text_field="text",
                    max_seq_length=max_seq_length,
                    args=UnslothTrainingArguments(**train_args),
                )

                stats = trainer.train()

                model = trainer.accelerator.unwrap_model(model, keep_fp32_wrapper=False)

                del trainer

            model = FastLanguageModel.for_inference(model)

            gc.collect()
            torch.cuda.empty_cache()

            memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
            print(f"[Rank {rank}] allocated {memory_allocated}MB for training")

            torch.cuda.reset_peak_memory_stats()

            print(f"[Rank {rank}] training stats for puzzle {key}: {stats}")

            puzzle_ds_multi = puzzle_ds.split_multi_replies()

            eval_ds = puzzle_ds_multi.augment(n=CFG["n_eval_aug"], seed=CFG["eval_aug_seed"])
            eval_ds = eval_ds.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)

            batches = make_view_batches(eval_ds, CFG["n_eval_aug"], CFG["decode_batch"])

            with torch.inference_mode():

                known_scores = {}

                for subkeys in batches:

                    spend_time = time.time() - start_time
                    if spend_time > CFG["task_cap"] or time.time() > end_time:
                        print(f"[Rank {rank}] timeout after {spend_time:.1f}s for puzzle {key}")
                        hit_cap = True
                        break

                    print(f"[Rank {rank}] decoding {subkeys}")

                    tokens = []
                    for subkey in subkeys:
                        data = eval_ds.get(subkey, formatter)
                        tokens.append(tokenizer.encode(data["input"]))

                    dfs_result = inference_turbo_dfs(model, tokens, max_new_tokens, max_score, end_time, CFG["dfs_window"])

                    for subkey_id, scored_beams in dfs_result:

                        subkey = subkeys[subkey_id]
                        bk = subkey.split(".")[0]
                        decoded_result = []

                        for beam_score, tokens_ in scored_beams:

                            array = formatter.convert_tokens_to_array(tokens_)
                            if array is None:
                                continue

                            solution = puzzle_ds_multi.invert_mod(array, subkey, inv_perm=True)

                            grid_id = (bk, tuple(map(tuple, solution)))

                            if grid_id in known_scores:
                                augmented_scores = known_scores[grid_id]
                            else:
                                print(f"[Rank {rank}] scoring {subkey} #{len(decoded_result)}")
                                aug_dataset = ArcDataset(
                                    keys=[bk],
                                    queries={bk: puzzle_ds_multi.queries.get(bk)},
                                    replies={bk: [solution.tolist()]},
                                )
                                aug_dataset = aug_dataset.augment(seed=stable_seed(bk, CFG["score_seed_off"]))
                                aug_dataset = aug_dataset.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)
                                aug_queries = []
                                aug_answers = []
                                for augmented_sample in aug_dataset.as_list(formatter):
                                    aug_queries.append(augmented_sample["input"])
                                    aug_answers.append(augmented_sample["reply"])
                                augmented_scores1 = calc_scores(aug_queries[:4], aug_answers[:4], tokenizer, model)
                                augmented_scores2 = calc_scores(aug_queries[4:], aug_answers[4:], tokenizer, model)
                                augmented_scores = augmented_scores1 + augmented_scores2
                                known_scores[grid_id] = augmented_scores

                            decoded_result.append({
                                "beam_score": beam_score,
                                "score_aug": augmented_scores,
                                "solution": solution,
                            })

                        if len(decoded_result):
                            with bz2.BZ2File(os.path.join(dir_outputs, subkey), "w") as f:
                                pickle.dump(decoded_result, f)

            memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
            print(f"[Rank {rank}] allocated {memory_allocated}MB for inference")

        except Exception as e:
            print(f"[Rank {rank}] ERROR on puzzle {key}: {type(e).__name__}: {e}")
            traceback.print_exc()
            try:
                model = FastLanguageModel.for_inference(model)
            except Exception:
                pass
            gc.collect()
            torch.cuda.empty_cache()
            if isinstance(e, torch.cuda.OutOfMemoryError):
                torch.cuda.synchronize()

        spend_time = time.time() - start_time
        _restore_task_overrides(applied)
        try:
            n_shards = len([f for f in os.listdir(dir_outputs) if f.startswith(key + "_")])
        except Exception:
            n_shards = -1
        _tel({"task": key, "pass": os.getenv("ARC_PASS_NAME", "?"),
              "elapsed": round(spend_time, 1), "hit_cap": hit_cap,
              "n_shards": n_shards, "overrides": _TASK_CFG.get(key, {})})
        print(f"[Rank {rank}] finished {key} in {spend_time:.1f}s")


In [ ]:
%%writefile starter.py
import os
import sys
import time
import json
import torch
import argparse
import traceback
import torch.multiprocessing as mp


def local_worker(rank, queue, end_time, test_path, marker_dir):

    os.environ["CUDA_VISIBLE_DEVICES"] = str(rank)

    torch.set_default_device("cpu")

    # Serialize the unsloth import/patching across workers (baseline behaviour),
    # but never wait forever for a dead predecessor.
    if rank > 0:
        waited = 0
        while not os.path.exists(os.path.join(marker_dir, f"worker{rank-1}")) and waited < 900:
            time.sleep(5)
            waited += 5

    from arc_solver import worker

    with open(os.path.join(marker_dir, f"worker{rank}"), "w") as f:
        f.write("Ok")

    print(f"[Rank {rank}] start!")

    attempts = 0
    while attempts < 2 and time.time() < end_time:
        attempts += 1
        try:
            worker(rank, queue, end_time, test_path=test_path)
            break
        except Exception as e:
            print(f"[Rank {rank}] worker crashed ({type(e).__name__}: {e}); attempt {attempts}")
            traceback.print_exc()
            try:
                import gc
                gc.collect()
                torch.cuda.empty_cache()
            except Exception:
                pass
            if attempts >= 2:
                print(f"[Rank {rank}] giving up.")

    print(f"[Rank {rank}] done!")


def estimated_work(task):
    """Rough cost proxy: tokens of all train pairs (TTT + context) plus an estimate of
    the decoded output sizes. Cheap tasks first => the deadline truncates the most
    expensive (and least likely to be solved) tail."""
    def ntok(g):
        return len(g) * (len(g[0]) + 1)
    train_tokens = sum(ntok(p["input"]) + ntok(p["output"]) for p in task["train"])
    ratios = [ntok(p["output"]) / max(1, ntok(p["input"])) for p in task["train"]]
    ratios.sort()
    ratio = ratios[len(ratios) // 2]
    test_tokens = sum(ntok(t["input"]) * (1 + ratio) for t in task["test"])
    return train_tokens * 16 + test_tokens * 8 * len(task["test"])


if __name__ == "__main__":

    parser = argparse.ArgumentParser()
    parser.add_argument("--end-time", type=float, default=0.0)
    parser.add_argument("--keys-file", type=str, default="")
    parser.add_argument("--nprocs", type=int, default=0)
    parser.add_argument("--order", type=str, default="cheap", choices=["cheap", "sorted", "file"])
    parser.add_argument("--test-path", type=str, default="")
    parser.add_argument("--marker-dir", type=str, default="/kaggle/working/markers")
    args, _ = parser.parse_known_args()

    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

    if args.test_path:
        test_path = args.test_path
    elif rerun_mode:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
    else:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"

    with open(test_path, "r") as f:
        data = json.load(f)

    if args.keys_file:
        with open(args.keys_file) as f:
            keys = [k for k in json.load(f) if k in data]
    else:
        keys = sorted(data.keys())
        if not rerun_mode and os.getenv("ARC_ALL_KEYS", "") != "1":
            debug_keys = os.getenv("ARC_DEBUG_KEYS", "0934a4d8,36a08778,981571dc,aa4ec2a5").split(",")
            keys = [k for k in keys if k in debug_keys]

    if args.order == "cheap":
        keys = sorted(keys, key=lambda k: estimated_work(data[k]))
    elif args.order == "sorted":
        keys = sorted(keys)

    nprocs = args.nprocs or min(4, torch.cuda.device_count())
    os.makedirs(args.marker_dir, exist_ok=True)
    for f_ in os.listdir(args.marker_dir):
        try:
            os.remove(os.path.join(args.marker_dir, f_))
        except Exception:
            pass

    print(f"[starter] {len(keys)} tasks, {nprocs} workers, order={args.order}, "
          f"budget={(args.end_time - time.time())/60:.1f} min, test_path={test_path}")

    queue = mp.Manager().Queue()
    for key in keys:
        queue.put(key)
    for _ in range(nprocs):
        queue.put(None)

    try:
        mp.spawn(local_worker, args=(queue, args.end_time, test_path, args.marker_dir), nprocs=nprocs)
    except Exception as e:
        print(f"[starter] spawn finished with error: {type(e).__name__}: {e}")
        traceback.print_exc()
    print("[starter] finished.")


In [ ]:

# ---------------------------------------------------------------------------
# Incremental submission checkpointing.
# A daemon thread re-derives /kaggle/working/submission.json from every candidate
# shard that exists RIGHT NOW, so the run always leaves a usable submission behind
# even if a later phase crashes or the platform kills the notebook at the limit.
# It only recomputes when the candidate stores actually changed, so the overhead
# across a 12 h run is negligible and it never competes with the GPU workers.
#
# v3: all three candidate stores (A, B, C) are folded into the checkpoint.
# ---------------------------------------------------------------------------
import os, time, json, threading, traceback

CKPT_INTERVAL = int(os.getenv("ARC_CKPT_INTERVAL", "240"))
# (store, run_name) -- run_name disambiguates sample keys from different sweeps.
if os.getenv("ARC_CKPT_C_STORE"):
    CKPT_STORES = [
        ("/kaggle/inference_outputs",   ""),
        ("/kaggle/inference_outputs_b", ".b"),
        (os.getenv("ARC_CKPT_C_STORE"), ".c"),
    ]
else:
    CKPT_STORES = [
        ("/kaggle/inference_outputs",   ""),
        ("/kaggle/inference_outputs_b", ".b"),
        ("/kaggle/inference_outputs_c", ".c"),
    ]

def _store_signature():
    sig = []
    for store, _ in CKPT_STORES:
        if not os.path.isdir(store):
            sig.append((store, -1, -1)); continue
        names = os.listdir(store)
        newest = 0
        for n in names:
            try:
                m = os.path.getmtime(os.path.join(store, n))
                newest = max(newest, m)
            except OSError:
                pass
        sig.append((store, len(names), int(newest)))
    return tuple(sig)

def _build_submission():
    from arc_loader import ArcDataset
    from arc_decoder import ArcDecoder

    RERUN = bool(os.getenv("KAGGLE_IS_COMPETITION_RERUN"))
    if RERUN:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
    else:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"

    data = ArcDataset.from_file(test_path)
    dm = data.split_multi_replies()
    dec = ArcDecoder(dm, n_guesses=2)
    any_store = False
    for store, run_name in CKPT_STORES:
        if os.path.isdir(store):
            try:
                dec.load_decoded_results(store, run_name=run_name)
                any_store = any_store or bool(dec.decoded_results)
            except Exception:
                traceback.print_exc()
    if not any_store:
        return None

    try:
        sel = dec.run_selection_algo()
    except Exception:
        traceback.print_exc()
        return None

    def _ok(g):
        return (isinstance(g, list) and len(g) > 0
                and all(isinstance(r, list) and len(r) == len(g[0]) and len(r) > 0 for r in g))

    sub = data.get_submission(sel)
    for k in data.keys:
        for i, t in enumerate(data.queries[k]["test"]):
            e = sub[k][i]
            for attempt in ("attempt_1", "attempt_2"):
                e[attempt] = [[int(x) for x in row] for row in e[attempt]] if _ok(e[attempt]) else [[0]]
            if not _ok(e["attempt_1"]) or e["attempt_1"] == [[0]]:
                e["attempt_1"] = [[int(x) for x in row] for row in t["input"]]
            if not _ok(e["attempt_2"]) or e["attempt_2"] == e["attempt_1"]:
                e["attempt_2"] = [[0]]
    return sub

_CKPT_STATE = {"sweeps": 0, "writes": 0, "last": "-"}

def _ckpt_loop():
    last_sig = None
    while True:
        ended = False
        try:
            end = globals().get("global_end_time", 0)
            if end and time.time() > end:
                ended = True
        except Exception:
            pass
        try:
            _CKPT_STATE["sweeps"] += 1
            sig = _store_signature()
            if sig != last_sig:
                sub = _build_submission()
                if sub is not None:
                    tmp = "/kaggle/working/submission.json.tmp"
                    with open(tmp, "w") as f:
                        json.dump(sub, f)
                    os.replace(tmp, "/kaggle/working/submission.json")
                    last_sig = sig
                    _CKPT_STATE["writes"] += 1
                    _CKPT_STATE["last"] = time.strftime("%H:%M:%S")
                    print("[ckpt] submission.json updated "
                          f"(write #{_CKPT_STATE['writes']}, {len(sub)} tasks, {_CKPT_STATE['last']})",
                          flush=True)
        except Exception:
            traceback.print_exc()
        try:
            json.dump(_CKPT_STATE, open("/kaggle/working/logs/ckpt_state.json", "w"))
        except Exception:
            pass
        if ended:
            print("[ckpt] budget elapsed; writer done.", flush=True)
            return
        time.sleep(CKPT_INTERVAL)

os.makedirs("/kaggle/working/logs", exist_ok=True)
threading.Thread(target=_ckpt_loop, daemon=True).start()
print(f"[ckpt] incremental checkpoint writer started over {len(CKPT_STORES)} stores "
      f"(sweep every {CKPT_INTERVAL}s)")


In [ ]:
# ---------------------------------------------------------------------------
# Pass A -- NVARC (perfpatch maths unchanged) in a half-cost configuration, cheap-first task
# order, one worker per L4. Per-task cap 800 s measured from task start (includes TTT).
#
# v3: pass A is capped at PHASE['A'] (45% of the wall clock) instead of the whole
# budget, so the two seeded diversity sweeps are guaranteed a share.  Any task
# pass A does not reach is picked up by the catch-up phase inside pass B using
# this exact configuration, so coverage is never lost.
# ---------------------------------------------------------------------------
import subprocess, sys, time, os
COMMON_ENV = {
    "UNSLOTH_DISABLE_STATISTICS": "1",
    "TRITON_PTXAS_PATH": "/usr/local/cuda/bin/ptxas",
    "OMP_NUM_THREADS": "12",
    "PYTHONHASHSEED": "0",
    "ARC_TASK_CAP": os.getenv("ARC_PRIMARY_TASK_CAP", "800"),
    # fast pass: 64 TTT sequences (8 aug x 8 geometries) and 8 decoded views instead of 128 / 16.
    # Public evidence (LB runtime board, Sep 2026): half-cost NVARC passes score in the same 34-36 band
    # (Nguyen 35.97 in 5h44, Mithil 33.89 in 3h50). The saved time buys two more independent passes.
    "ARC_N_TRAIN_AUG": os.getenv("ARC_N_TRAIN_AUG", "8"),
    "ARC_N_EVAL_AUG": os.getenv("ARC_N_EVAL_AUG", "1"),
}
os.environ.update(COMMON_ENV)
os.environ["ARC_OUT_DIR"] = "/kaggle/inference_outputs"
os.environ["ARC_PASS_NAME"] = "A"
os.environ["ARC_TELEMETRY_FILE"] = "/kaggle/working/logs/tel_A.jsonl"
t_a = time.time()
rc = subprocess.call([sys.executable, "starter.py", "--end-time", f"{PHASE['A']}", "--order", "cheap"])
print(f"pass-A rc={rc} took {(time.time()-t_a)/60:.1f} min; remaining {(global_end_time-time.time())/60:.1f} min")


In [ ]:
# ---------------------------------------------------------------------------
# Pass B -- with the budget up to PHASE['B']: (1) catch-up of tasks pass A never
# reached (same config, into the pass-A store), then (2) an independent second
# pass over ALL tasks with different seeds (cheap-first, truncated by the phase
# deadline). Pass A outputs are never modified.
# ---------------------------------------------------------------------------
import os, sys, json, time, subprocess
from arc_loader import ArcDataset
from arc_decoder import ArcDecoder
from starter import estimated_work

def remaining():
    return global_end_time - time.time()

test_path = ("/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json" if RERUN
             else "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json")
data = ArcDataset.from_file(test_path)
keys_in_scope = data.keys if (RERUN or VALIDATE_FULL) else [k for k in data.keys if k in os.getenv("ARC_DEBUG_KEYS", "0934a4d8,36a08778,981571dc,aa4ec2a5").split(",")]

dec = ArcDecoder(data.split_multi_replies(), n_guesses=2)
dec.load_decoded_results("/kaggle/inference_outputs")
stats = dec.candidate_stats()
unprocessed = [k for k in keys_in_scope if not any(f"{k}_{i}" in stats for i in range(len(data.queries[k]["test"])))]
print(f"[pass-B] pass-A reached {len(keys_in_scope)-len(unprocessed)}/{len(keys_in_scope)} tasks; "
      f"remaining {remaining()/60:.1f} min; phase-B deadline in {(PHASE['B']-time.time())/60:.1f} min")

def run_pass(name, keys, env_overrides, out_dir, end_time, min_minutes=15):
    if not keys or (end_time - time.time()) < min_minutes * 60:
        print(f"[pass-B:{name}] skipped (keys={len(keys)}, phase window={(end_time-time.time())/60:.1f} min)")
        return
    keys_file = f"/kaggle/working/keys_{name}.json"
    json.dump(keys, open(keys_file, "w"))
    env = dict(os.environ, ARC_OUT_DIR=out_dir, ARC_PASS_NAME=name,
               ARC_TELEMETRY_FILE=f"/kaggle/working/logs/tel_{name}.jsonl",
               **{k: str(v) for k, v in env_overrides.items()})
    t = time.time()
    rc = subprocess.call([sys.executable, "starter.py", "--end-time", f"{end_time}", "--keys-file", keys_file, "--order", "file"], env=env)
    print(f"[pass-B:{name}] rc={rc} keys={len(keys)} took {(time.time()-t)/60:.1f} min; remaining {remaining()/60:.1f} min")

cheap_first = lambda ks: sorted(ks, key=lambda k: estimated_work(data.queries[k]))

# (1) catch-up with the pass-A configuration into the pass-A store
run_pass("catchup", cheap_first(unprocessed), {}, "/kaggle/inference_outputs", PHASE["B"])

# (2) independent second pass over all tasks: new LoRA-init / augmentation / scoring seeds
SEED_B = dict(ARC_LORA_SEED=137, ARC_TRAIN_AUG_SEED=17, ARC_EVAL_AUG_SEED=29, ARC_SCORE_SEED_OFFSET=7,
              ARC_N_TRAIN_AUG=8, ARC_N_EVAL_AUG=1, ARC_TASK_CAP=700)
run_pass("seedB", cheap_first(keys_in_scope), SEED_B, "/kaggle/inference_outputs_b", PHASE["B"])
json.dump({"unprocessed_after_A": unprocessed, "seed_B": SEED_B}, open("/kaggle/working/passB_plan.json", "w"))


In [ ]:
# ---------------------------------------------------------------------------
# Pass C, part 1 (v4) -- coverage repair, unchanged from v3.
#
# (a) catchup2: every task A and B never reached is processed with the pass-A
#     configuration into the pass-A store, so the controller below always has
#     a defined diagnostic vector for every task (coverage first -- the v3
#     principle is kept verbatim).
# The controller cell runs next; the arms run in part 2.
# ---------------------------------------------------------------------------
import os, sys, json, time, subprocess
from arc_loader import ArcDataset
from arc_decoder import ArcDecoder
from starter import estimated_work

test_path = ("/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json" if RERUN
             else "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json")
data = ArcDataset.from_file(test_path)
keys_in_scope = data.keys if (RERUN or VALIDATE_FULL) else [k for k in data.keys if k in os.getenv("ARC_DEBUG_KEYS", "0934a4d8,36a08778,981571dc,aa4ec2a5").split(",")]

dec = ArcDecoder(data.split_multi_replies(), n_guesses=2)
dec.load_decoded_results("/kaggle/inference_outputs")
dec.load_decoded_results("/kaggle/inference_outputs_b", run_name=".b")
stats = dec.candidate_stats()
unprocessed = [k for k in keys_in_scope if not any(f"{k}_{i}" in stats for i in range(len(data.queries[k]["test"])))]
print(f"[pass-C] A+B reached {len(keys_in_scope)-len(unprocessed)}/{len(keys_in_scope)} tasks; "
      f"remaining {(global_end_time-time.time())/60:.1f} min")

def run_pass_c(name, keys, env_overrides, out_dir, end_time, min_minutes=12):
    if PROFILE == "commit":
        min_minutes = 1
    if not keys or (end_time - time.time()) < min_minutes * 60:
        print(f"[pass-C:{name}] skipped (keys={len(keys)}, phase window={(end_time-time.time())/60:.1f} min)")
        return
    keys_file = f"/kaggle/working/keys_{name}.json"
    json.dump(keys, open(keys_file, "w"))
    env = dict(os.environ, ARC_OUT_DIR=out_dir, ARC_PASS_NAME=name,
               ARC_TELEMETRY_FILE=f"/kaggle/working/logs/tel_{name}.jsonl",
               **{k: str(v) for k, v in env_overrides.items()})
    t = time.time()
    rc = subprocess.call([sys.executable, "starter.py", "--end-time", f"{end_time}", "--keys-file", keys_file, "--order", "file"], env=env)
    print(f"[pass-C:{name}] rc={rc} keys={len(keys)} took {(time.time()-t)/60:.1f} min; remaining {(global_end_time-time.time())/60:.1f} min")

cheap_first = lambda ks: sorted(ks, key=lambda k: estimated_work(data.queries[k]))

# (a) coverage repair with the pass-A configuration (v3 behaviour, byte-identical config)
run_pass_c("catchup2", cheap_first(unprocessed), {}, "/kaggle/inference_outputs", PHASE["C_catchup"])


In [ ]:
# ---------------------------------------------------------------------------
# v5 CONTROLLER (theory v32 landing, footprint-matched revision)
#
# Signals phi are computed from the pass-A / pass-B / catchup candidate pools
# ONLY (zero extra GPU).  The rule mu, the intervention table and the cost
# table are frozen in ARC-AGI-2-v5-controller-prereg.md (sections 2-3) and
# are not edited after seeing any run result.
#
# WHAT CHANGED FROM v4 AND WHY (evidence: v4 A/B, kernel_out_v4_ab):
#   * v4's per-task override keys were ALL decode-side (task_cap, n_eval_aug,
#     min_prob, dfs_window, score_seed_off, eval_aug_seed).  But the largest
#     diagnosed population -- express == 0 (37/120 tasks, no candidate at all
#     after A+B) -- is MODEL-side bottlenecked (the TTT artefact found
#     nothing).  v4's more_views was a footprint mismatch: decode-side effort
#     spent on a model-side problem (3 wins / 61 units, 95% miss).
#   * v5 unlocks the model side: train_aug_seed (and n_train_aug) join the
#     overridable keys, so an empty pool gets a *different trained model*
#     (training-augmentation rotation), not just more views of the same one.
#   * diversify (52/120 tasks, the modal v4 action) went 0 wins / 1 loss:
#     decode-side decorrelation on rich pools perturbs a selection that was
#     already working.  Retired: identify >= 4 now maps to standard (exact
#     v3 behaviour -- intervene only where the diagnosis says the default is
#     mismatched).
#
# Outputs (all under ARC_WORK_DIR, default /kaggle/working):
#   task_config.json      per-task CFG overrides consumed by the adaptive arm
#   logs/controller.json  full phi/mu record (telemetry for the paper)
#   logs/adaptive_order.json  diagnosis-weighted task order (attention
#                         reallocation is part of the intervention)
# ---------------------------------------------------------------------------
import os, json, time, zlib
from arc_loader import ArcDataset
from arc_decoder import ArcDecoder, getter_kgmon
from starter import estimated_work

WORK = os.getenv("ARC_WORK_DIR", "/kaggle/working")
STORE_A = os.getenv("ARC_STORE_A", "/kaggle/inference_outputs")
STORE_B = os.getenv("ARC_STORE_B", "/kaggle/inference_outputs_b")
TEL_DIR = os.getenv("ARC_TEL_DIR", os.path.join(WORK, "logs"))
test_path = os.getenv("ARC_TEST_PATH") or (
    "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json" if RERUN
    else "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json")
data = ArcDataset.from_file(test_path)
keys_all = data.keys if (RERUN or VALIDATE_FULL) else [k for k in data.keys
           if k in os.getenv("ARC_DEBUG_KEYS", "0934a4d8,36a08778,981571dc,aa4ec2a5").split(",")]

# --- candidate pools: A-store (+catchup, run_name "") and B-store (".b") -----
pool = ArcDecoder(data.split_multi_replies(), n_guesses=2)
pool.load_decoded_results(STORE_A)
pool.load_decoded_results(STORE_B, run_name=".b")

# --- search-pressure signal: cap hits recorded by the workers (telemetry) ----
hit = set()
for _name in ("A", "catchup", "seedB", "catchup2"):
    _p = os.path.join(TEL_DIR, f"tel_{_name}.jsonl")
    if os.path.isfile(_p):
        for _line in open(_p):
            try:
                _rec = json.loads(_line)
            except Exception:
                continue
            if _rec.get("hit_cap"):
                hit.add(_rec["task"])

# --- frozen intervention table (prereg v2 section 3; base seed-C is 53/71/23) -
# The pass-C base training-augmentation seed is 53 (SEED_C).  A rotated seed is
# derived per task from a stable hash (zlib.crc32 -- process-independent, the
# dhash lesson) into [100, 999]: disjoint from {1, 17, 53} and from each other
# with high probability, so every empty-pool task gets its own model-side bet
# while remaining bit-reproducible across processes and runs.
def _rotated_train_seed(task):
    return 100 + (zlib.crc32(task.encode("utf-8")) % 900)

OVERRIDES = {
    "deepen":      {"min_prob": 0.15, "dfs_window": 700, "task_cap": 1000},
    "verify_more": {"score_seed_off": 91, "task_cap": 750},
    "light":       {"task_cap": 450},
    "standard":    {},
}
GROUP_ORDER = ["reseed", "deepen", "verify_more", "light", "standard"]

def _phi(split_key):
    """Diagnostic vector for one test input (prereg section 2)."""
    samples = pool.decoded_results.get(split_key)
    if not samples:
        return {"express": 0, "verify": 0.0, "identify": 0}
    groups = {}
    for g in samples.values():
        groups.setdefault(tuple(map(tuple, g["solution"])), []).append(g)
    scored = sorted(((getter_kgmon(gs), h) for h, gs in groups.items()),
                    key=lambda x: x[0], reverse=True)
    uniq6 = len({h for _, h in scored[:6]})
    if len(scored) > 1:
        margin = scored[0][0] - scored[1][0]
        verify = margin / max(abs(scored[0][0]), 1e-9)
    else:
        verify = float("inf")          # single candidate: nothing to verify against
    return {"express": len(groups), "verify": verify, "identify": uniq6}

def _mu(phi):
    """Frozen decision rule (prereg v2 section 3); order matters.

    Footprint-matching principle: the intervention must act on the side where
    the bottleneck is.  express == 0 is a model-side bottleneck (reseed);
    a cap hit is a search-side bottleneck (deepen); a tiny top-2 margin is a
    selection-side bottleneck (verify_more).  The retired v4 diversify
    population (identify >= 4, a rich pool with a confident margin) is NOT a
    diagnosed mismatch, so it returns to the exact v3 default (standard) --
    the controller intervenes only where the default is mismatched.  The
    comfortable middle (express >= 2, verify >= 0.02, identify < 4) keeps
    v4's light sweep."""
    if phi["express"] == 0:
        return "reseed"           # model-side: rotate the training augmentation
    if phi["search"]:
        return "deepen"           # search-side: buy more search
    if 0 < phi["verify"] < 0.02:
        return "verify_more"      # selection-side: independent scoring stream
    if phi["express"] >= 2 and phi["verify"] >= 0.02 and phi["identify"] < 4:
        return "light"            # comfortable middle: cheap sweep
    return "standard"             # rich pool / single candidate: exact v3 behaviour

records, configs, by_action = {}, {}, {a: [] for a in GROUP_ORDER}
for k in keys_all:
    subs = [f"{k}_{i}" for i in range(len(data.queries[k]["test"]))]
    phis = []
    for s in subs:
        p = _phi(s)
        p["search"] = k in hit
        phis.append(p)
    agg = {"express": min(p["express"] for p in phis),
           "search": any(p["search"] for p in phis),
           "verify": min(p["verify"] for p in phis),
           "identify": max(p["identify"] for p in phis)}
    action = _mu(agg)
    rec_phi = {}
    for kk, vv in agg.items():
        rec_phi[kk] = ("inf" if vv == float("inf") else
                       (round(vv, 4) if isinstance(vv, float) else vv))
    records[k] = {"phi": rec_phi, "action": action}
    if action == "reseed":
        # model-side bet + decode-side effort (v4's more_views package with
        # the new training rotation added)
        configs[k] = {"train_aug_seed": _rotated_train_seed(k),
                      "n_eval_aug": 2, "min_prob": 0.15, "task_cap": 950}
    else:
        configs[k] = OVERRIDES[action]
    by_action[action].append(k)

cheap_first = lambda ks: sorted(ks, key=lambda k: estimated_work(data.queries[k]))
adaptive_order = [k for a in GROUP_ORDER for k in cheap_first(by_action[a])]

os.makedirs(os.path.dirname(os.path.join(WORK, "task_config.json")), exist_ok=True)
json.dump(configs, open(os.path.join(WORK, "task_config.json"), "w"), indent=1)
json.dump({"profile": PROFILE, "records": records, "order": adaptive_order,
           "n_by_action": {a: len(v) for a, v in by_action.items()}},
          open(os.path.join(TEL_DIR, "controller.json"), "w"), indent=1)
json.dump(adaptive_order, open(os.path.join(TEL_DIR, "adaptive_order.json"), "w"))

print(f"[controller] {len(keys_all)} tasks; pools: {len(pool.decoded_results)} "
      f"split-keys with candidates; cap-hit tasks: {len(hit)}")
print("[controller] mu decisions: " + ", ".join(f"{a}={len(by_action[a])}" for a in GROUP_ORDER))
print(f"[controller] wrote task_config.json ({len(configs)} entries) and logs/controller.json")


In [ ]:
# ---------------------------------------------------------------------------
# Pass C, part 2 (v4) -- the arms.
#
#   ab_validate : fixed arm (exact v3 seed-C config, cheap-first) until
#                 T_FIXED, then the adaptive arm (per-task mu configs,
#                 diagnosis-weighted order) until the global end.  Equal
#                 wall-clock window (45/55 split), separate stores.
#   submission  : the single pre-registered arm over the full C window into
#                 the v3 store (/kaggle/inference_outputs_c).
#
# Safety invariants (prereg section 5): pass-A/B stores are never written
# here; coverage repair already ran; selection keeps pass-A top-1.
# ---------------------------------------------------------------------------
import os, sys, json, time, subprocess

SEED_C = dict(ARC_LORA_SEED=911, ARC_TRAIN_AUG_SEED=53, ARC_EVAL_AUG_SEED=71, ARC_SCORE_SEED_OFFSET=23,
              ARC_N_TRAIN_AUG=8, ARC_N_EVAL_AUG=1, ARC_TASK_CAP=650)
adaptive_order = json.load(open("/kaggle/working/logs/adaptive_order.json"))
CFG_PATH = "/kaggle/working/task_config.json"

if PROFILE == "ab_validate":
    # (b1) fixed arm: the exact v3 seed-C sweep, into its own store
    run_pass_c("cfixed", cheap_first(keys_in_scope), SEED_C,
               "/kaggle/inference_outputs_c_fixed", T_FIXED)
    # (b2) adaptive arm: mu per-task configs + diagnosis-weighted order
    run_pass_c("cadaptive", adaptive_order,
               dict(SEED_C, ARC_TASK_CONFIG_FILE=CFG_PATH),
               "/kaggle/inference_outputs_c_adaptive", PHASE["C"])
    json.dump({"unprocessed_after_AB": unprocessed, "seed_C": SEED_C,
               "arms": ["cfixed", "cadaptive"], "window_split": "45/55"},
              open("/kaggle/working/passC_plan.json", "w"))
else:
    if SUBMIT_ARM == "adaptive":
        run_pass_c("cadaptive", adaptive_order,
                   dict(SEED_C, ARC_TASK_CONFIG_FILE=CFG_PATH),
                   "/kaggle/inference_outputs_c", PHASE["C"])
    else:
        run_pass_c("cfixed", cheap_first(keys_in_scope), SEED_C,
                   "/kaggle/inference_outputs_c", PHASE["C"])
    json.dump({"unprocessed_after_AB": unprocessed, "seed_C": SEED_C, "submit_arm": SUBMIT_ARM},
              open("/kaggle/working/passC_plan.json", "w"))


In [ ]:
# ---------------------------------------------------------------------------
# Selection (v6s): v3 logic + P4 cross-group second candidate.
#
# THEORY BASIS (v32-core section 6 / section 12, Campaign B, pre-registered
# route B1): on identify-heavy synthetic families the second attempt should
# cover a DIFFERENT rule interpretation rather than a marginal perturbation of
# the first (proposition D2).  Measured: Delta_cov = +0.0410,
# 95% CI [+0.0070, +0.0750], n = 1,000 disagreement tasks -- established on the
# synthetic layer; the real-ARC effect is measured by THIS run (secondary
# analysis, clearly separated from the pre-registered controller verdict).
#
# P4 CHANGE (attempt_2 fallback path only): when the independent probmul
# hedge does not yield a distinct candidate, prefer a STRUCTURALLY DISTINCT
# candidate -- different output shape, or > 25% of cells different, a proxy for
# "different rule family" -- over the marginal runner-up, which is typically a
# parameter perturbation of attempt_1 (same rule family, highly correlated
# errors).  If no structurally distinct candidate exists in the top-8 unique
# solutions, fall back to the exact v3 behaviour (first different in kg[1:4]).
# attempt_1, the probmul hedge, and the pass-A top-1 invariant are unchanged.
#
# In ab_validate mode BOTH selection variants are scored on BOTH arms:
#   primary   (pre-registered, comparable to the v4 A/B): v3 selection,
#             adaptive vs fixed -- the controller verdict.
#   secondary (exploratory): the P4 delta on each arm (McNemar exact p).
# submission.json = adaptive arm + P4 selection (the v6 submission semantics).
# ---------------------------------------------------------------------------
import os, json, hashlib, math, random
import numpy as np
from arc_loader import ArcDataset
from arc_decoder import ArcDecoder, score_kgmon, score_full_probmul_3

WORK = os.getenv("ARC_WORK_DIR", "/kaggle/working")
test_path = os.getenv("ARC_TEST_PATH") or (
    "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json" if RERUN
    else "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json")
data = ArcDataset.from_file(test_path)
if not RERUN:
    data = data.load_replies("/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_solutions.json")
dm = data.split_multi_replies()

def _same(x, y):
    return x is not None and y is not None and np.array_equal(np.asarray(x), np.asarray(y))

def _distinct_enough(a, b, thresh=0.25):
    """Structural-distinctness proxy for 'different rule family' (P4 / D2)."""
    A, B = np.asarray(a), np.asarray(b)
    if A.shape != B.shape:
        return True
    if A.size == 0:
        return False
    return float(np.mean(A != B)) > thresh

def select_two(sel_kg, sel_pm, sel_a, p4=True):
    """v3 selection logic; p4=True adds the structurally-distinct second
    candidate (v6).  Returns (selected, n_hedged, n_forced, n_p4)."""
    selected, n_hedged, n_forced, n_p4 = {}, 0, 0, 0
    for bk in set(sel_kg) | set(sel_pm) | set(sel_a):
        kg = list(sel_kg.get(bk, []))
        pm = list(sel_pm.get(bk, []))
        a1 = (sel_a.get(bk) or [None])[0]
        first = kg[0] if kg else (pm[0] if pm else None)
        top = [first] if first is not None else []
        second = None
        # attempt_2, step 1: hedge with the other scorer family (v3, unchanged)
        for cand in pm[:2]:
            if not _same(cand, first):
                second = cand
                break
        if second is not None:
            n_hedged += 1
        else:
            # attempt_2, step 2 (v6/P4): structurally distinct candidate first
            if p4 and first is not None:
                for cand in kg[1:8]:
                    if not _same(cand, first) and _distinct_enough(cand, first):
                        second = cand
                        n_p4 += 1
                        break
            # attempt_2, step 3: exact v3 fallback (best-scoring different)
            if second is None:
                for cand in kg[1:4]:
                    if not _same(cand, first):
                        second = cand
                        break
        if second is not None:
            top.append(second)
        # invariant: pass A's top-1 is always among the two attempts
        if a1 is not None and not any(_same(a1, t) for t in top):
            top = (top[:1] + [a1]) if top else [a1]
            n_forced += 1
        selected[bk] = top[:2]
    return selected, n_hedged, n_forced, n_p4

def finalize(selected):
    """v3 fallback/repair logic, unchanged."""
    submission = data.get_submission(selected)
    n_fallback = 0
    for k in data.keys:
        for i, t in enumerate(data.queries[k]["test"]):
            e = submission[k][i]
            def ok(g):
                return isinstance(g, list) and len(g) > 0 and all(isinstance(r, list) and len(r) == len(g[0]) and len(r) > 0 for r in g)
            if not ok(e["attempt_1"]) or e["attempt_1"] == [[0]]:
                e["attempt_1"] = [[int(x) for x in row] for row in t["input"]]
                n_fallback += 1
            if not ok(e["attempt_2"]) or e["attempt_2"] == e["attempt_1"]:
                e["attempt_2"] = [[0]]
            e["attempt_1"] = [[int(x) for x in row] for row in e["attempt_1"]]
            e["attempt_2"] = [[int(x) for x in row] for row in e["attempt_2"]]
    return submission, n_fallback

def _mcnemar(units):
    """units: list of (key, i, hit_fixed, hit_adaptive) -> (b, c, p_two_exact)."""
    b = sum(1 for _, _, sf, sa in units if sf and not sa)
    c = sum(1 for _, _, sf, sa in units if sa and not sf)
    n_d = b + c
    if n_d:
        kk = min(b, c)
        tail = sum(math.comb(n_d, j) for j in range(0, kk + 1)) / (2 ** n_d)
        p_two = min(1.0, 2 * tail)
    else:
        p_two = 1.0
    return b, c, p_two

# --- pass A alone: the monotone safety reference (shared by both arms) ------
dec_a = ArcDecoder(dm, n_guesses=2)
dec_a.load_decoded_results(os.getenv("ARC_STORE_A", "/kaggle/inference_outputs"))
sel_a = dec_a.run_selection_algo(score_kgmon)

if PROFILE == "ab_validate":
    ARMS = {
        "fixed":    [(os.getenv("ARC_STORE_A", "/kaggle/inference_outputs"), ""),
                     ("/kaggle/inference_outputs_b", ".b"),
                     ("/kaggle/inference_outputs_c_fixed", ".cf")],
        "adaptive": [(os.getenv("ARC_STORE_A", "/kaggle/inference_outputs"), ""),
                     ("/kaggle/inference_outputs_b", ".b"),
                     ("/kaggle/inference_outputs_c_adaptive", ".ca")],
    }
    pools, sels = {}, {}
    for arm, stores in ARMS.items():
        pooled = ArcDecoder(dm, n_guesses=2)
        n_samples = 0
        for store, run_name in stores:
            n_samples += pooled.load_decoded_results(store, run_name=run_name)
        pools[arm] = pooled
        sels[arm] = (pooled.run_selection_algo(score_kgmon),
                     pooled.run_selection_algo(score_full_probmul_3))
        print(f"*** [{arm}] pool samples={n_samples}")

    # ---- 2x2: {fixed, adaptive} x {v3 selection, P4 selection} -------------
    result, subs, subs_v3 = {}, {}, {}
    for arm in ARMS:
        for variant, p4 in (("v3", False), ("P4", True)):
            selected, n_hedged, n_forced, n_p4 = select_two(sels[arm][0], sels[arm][1], sel_a, p4=p4)
            submission, n_fallback = finalize(selected)
            R = data.validate_submission(submission)
            result[f"{arm}_{variant}"] = {"R": R, "hedged": n_hedged, "forced": n_forced,
                                          "p4": n_p4, "fallback": n_fallback}
            print(f"*** [{arm}/{variant}] R = {R:.4f} (hedged {n_hedged}, forced {n_forced}, "
                  f"p4 {n_p4}, fallback {n_fallback})")
            json.dump(submission, open(os.path.join(WORK, f"submission_{arm}_{variant}.json"), "w"))
            if variant == "v3":
                subs_v3[arm] = submission
            else:
                subs[arm] = submission
    json.dump(subs["adaptive"], open(os.path.join(WORK, "submission.json"), "w"))
    json.dump(subs["fixed"], open(os.path.join(WORK, "submission_fixed.json"), "w"))

    # ---- primary verdict: controller (v3 selection, pre-registered) --------
    def _hits(sub, k, i, r):
        return any(np.array_equal(r, sub[k][i][f"attempt_{j}"]) for j in (1, 2))

    units = []
    for k in data.keys:
        for i, r in enumerate(data.replies[k]):
            units.append((k, i, _hits(subs_v3["fixed"], k, i, r), _hits(subs_v3["adaptive"], k, i, r)))
    b, c, p_two = _mcnemar(units)
    Rf = sum(sf for _, _, sf, _ in units) / len(units)
    Ra = sum(sa for _, _, _, sa in units) / len(units)
    rng = random.Random(20261017)
    deltas = []
    for _ in range(10000):
        samp = [units[rng.randrange(len(units))] for _ in range(len(units))]
        deltas.append(sum(sf for _, _, sf, _ in samp) / len(samp)
                      - sum(sa for _, _, _, sa in samp) / len(samp))
    deltas.sort()
    lo, hi = deltas[int(0.025 * len(deltas))], deltas[int(0.975 * len(deltas)) - 1]
    d = Rf - Ra
    if p_two > 0.05:
        verdict = "not_supported"
    elif d <= 0:
        verdict = "supported"
    else:
        verdict = "refuted"
    print(f"*** A/B (primary, v3 selection): R_fixed={Rf:.4f} R_adaptive={Ra:.4f} "
          f"Delta_adaptive={d:+.4f} bootstrap 95% CI [{lo:+.4f}, {hi:+.4f}]")
    print(f"*** McNemar: b(fixed only)={b} c(adaptive only)={c} exact two-sided p={p_two:.4f}")
    print(f"*** H-match on real ARC (pre-registered line): {verdict}")

    # ---- secondary: P4 delta per arm (exploratory, McNemar exact) ----------
    p4_secondary = {}
    for arm in ("fixed", "adaptive"):
        u2 = []
        for k in data.keys:
            for i, r in enumerate(data.replies[k]):
                s_v3 = _hits(subs_v3[arm], k, i, r)
                s_p4 = _hits(subs[arm], k, i, r)
                u2.append((k, i, s_v3, s_p4))
        b2, c2, p2 = _mcnemar(u2)
        Rv3 = sum(x for _, _, x, _ in u2) / len(u2)
        Rp4 = sum(x for _, _, _, x in u2) / len(u2)
        p4_secondary[arm] = {"R_v3": Rv3, "R_P4": Rp4, "Delta_P4": Rv3 - Rp4,
                             "mcnemar": {"b_v3_only": b2, "c_P4_only": c2, "p": p2}}
        print(f"*** [P4 secondary/{arm}] R_v3={Rv3:.4f} R_P4={Rp4:.4f} "
              f"Delta_P4={Rv3-Rp4:+.4f} McNemar b={b2} c={c2} p={p2:.4f}")

    json.dump({"R_fixed": Rf, "R_adaptive": Ra, "Delta_adaptive": d,
               "ci": [lo, hi], "mcnemar": {"b": b, "c": c, "p": p_two},
               "verdict": verdict, "n_units": len(units), "per_arm": result,
               "p4_secondary": p4_secondary,
               "units": [{"task": k, "i": i, "fixed": sf, "adaptive": sa}
                         for k, i, sf, sa in units]},
              open(os.path.join(WORK, "logs", "ab_result.json"), "w"), indent=1)
    print(f"*** submission.json (adaptive+P4) sha256="
          f"{hashlib.sha256(open(os.path.join(WORK, 'submission.json'), 'rb').read()).hexdigest()[:12]}")
else:
    pooled = ArcDecoder(dm, n_guesses=2)
    pooled.load_decoded_results("/kaggle/inference_outputs")
    n_b = pooled.load_decoded_results("/kaggle/inference_outputs_b", run_name=".b")
    n_c = pooled.load_decoded_results("/kaggle/inference_outputs_c", run_name=".c")
    print(f"*** pooled pool sizes: pass-B samples={n_b}, pass-C samples={n_c}")
    sel_kg = pooled.run_selection_algo(score_kgmon)
    sel_pm = pooled.run_selection_algo(score_full_probmul_3)
    selected, n_hedged, n_forced, n_p4 = select_two(sel_kg, sel_pm, sel_a, p4=True)
    submission, n_fallback = finalize(selected)
    print(f"*** pooled selection (v6, P4): {len(selected)} outputs, attempt_2 hedged by the independent "
          f"scorer on {n_hedged} outputs, structurally-distinct (P4) on {n_p4}, "
          f"pass-A top-1 forced back on {n_forced}")
    with open("/kaggle/working/submission.json", "w") as f:
        json.dump(submission, f)
    chk = json.load(open("/kaggle/working/submission.json"))
    assert set(chk) == set(data.keys)
    for k in data.keys:
        assert len(chk[k]) == len(data.queries[k]["test"]) and all("attempt_1" in e and "attempt_2" in e for e in chk[k])
    print(f"*** submission.json OK: {len(chk)} tasks, {sum(len(v) for v in chk.values())} outputs, "
          f"{n_fallback} identity fallbacks, "
          f"sha256={hashlib.sha256(open('/kaggle/working/submission.json','rb').read()).hexdigest()[:12]}")
    if not RERUN:
        print("--- pass A alone ---"); dec_a.benchmark_selection_algos()
        print("--- pooled A+B+C ---"); pooled.benchmark_selection_algos()
        print("*** Reload score (tasks):", data.validate_submission(chk))


In [ ]:

# ---------------------------------------------------------------------------
# Final safety net: make sure /kaggle/working/submission.json exists, covers every
# task in the active test set, is schema-valid, and is JSON-loadable.
# ---------------------------------------------------------------------------
import os, json, time, traceback

SUB = "/kaggle/working/submission.json"

def _ok(g):
    return (isinstance(g, list) and len(g) > 0
            and all(isinstance(r, list) and len(r) == len(g[0]) and len(r) > 0 for r in g))

RERUN_F = bool(os.getenv("KAGGLE_IS_COMPETITION_RERUN"))
TEST_F = ("/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json" if RERUN_F
          else "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json")
data_f = json.load(open(TEST_F))

try:
    sub = json.load(open(SUB))
except Exception:
    traceback.print_exc()
    sub = {}

if set(sub.keys()) != set(data_f.keys()):
    print(f"[final] rebuilding: submission had {len(sub)} keys, test set has {len(data_f)}")
    built = None
    try:
        built = _build_submission()
    except Exception:
        traceback.print_exc()
    sub = built if built is not None else {}

# fill / repair anything missing
for k, task in data_f.items():
    n = len(task["test"])
    if k not in sub or len(sub[k]) != n:
        sub[k] = [{"attempt_1": [], "attempt_2": []} for _ in range(n)]
    for i, t in enumerate(task["test"]):
        e = sub[k][i]
        inp = [[int(x) for x in row] for row in t["input"]]
        a1 = e.get("attempt_1", [[0]])
        a2 = e.get("attempt_2", [[0]])
        e["attempt_1"] = a1 if _ok(a1) else inp
        if not _ok(a2) or a2 == e["attempt_1"]:
            e["attempt_2"] = [[0]]
        e["attempt_1"] = [[int(x) for x in r] for r in e["attempt_1"]]
        e["attempt_2"] = [[int(x) for x in r] for r in e["attempt_2"]]

# drop stray keys
for k in list(sub.keys()):
    if k not in data_f:
        del sub[k]

tmp = SUB + ".final"
with open(tmp, "w") as f:
    json.dump(sub, f)
os.replace(tmp, SUB)

chk = json.load(open(SUB))
assert set(chk.keys()) == set(data_f.keys()), "submission keys mismatch"
for k, task in data_f.items():
    assert len(chk[k]) == len(task["test"]), f"{k}: wrong number of outputs"
    for e in chk[k]:
        assert _ok(e["attempt_1"]) and _ok(e["attempt_2"]), f"{k}: invalid grid"
print(f"[final] submission.json OK — {len(chk)} tasks, wrote at {time.strftime('%H:%M:%S')}")
print(f"[final] size = {os.path.getsize(SUB)} bytes")
try:
    print("[final] checkpoint writer stats:", _CKPT_STATE)
except Exception:
    pass
